# Calendar Agent — Colab LoRA 학습 (stable-fp16)

Kaggle 대안. **데이터가 repo에 포함**돼 있어 clone만으로 학습 가능 (Kaggle 데이터셋 불필요).

## 준비
- 우측 상단 **런타임 → 런타임 유형 변경 → GPU (T4)**
- 아래 셀을 위→아래로 실행. cell 1에서 GitHub PAT 입력.

## 정밀도
- `train.yaml`: bf16=false, fp16=true → **stable-fp16** (fp32 마스터 로드 + fp16 AMP). T4 텐서코어로 빠름(~8s/step) + 안정.
- 라운드는 `output_dir`에서 자동 인식.

In [ ]:
# 1. repo clone (private → PAT). 이미 있으면 최신으로 force-sync
import os, getpass
%cd /content
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None
else:
    !cd calendar-agent && git fetch origin && git reset --hard origin/main
%cd /content/calendar-agent
!git log --oneline -1

In [ ]:
# 2. 설치 + 환경 (wandb off; 단일 GPU는 train_lora.py가 강제)
!pip install -q -e .[train]
import os
os.environ['WANDB_DISABLED'] = 'true'
import torch
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

In [ ]:
# 3. 데이터 + 정밀도 확인 (데이터는 repo에 포함돼 clone으로 따라옴)
import yaml
for p in ['data/processed/train.jsonl', 'data/processed/val.jsonl', 'data/eval/golden.jsonl']:
    n = sum(1 for l in open(p, 'rb') if l.strip())
    print(f'{p}: {n} records')
c = yaml.safe_load(open('configs/train_colab.yaml'))
print(f"precision: bf16={c['bf16']} fp16={c['fp16']} | output_dir={c['output_dir']}")

In [ ]:
# 4. 학습 (cell 12 상응). 시작 시 '[train] ... -> fp16 (config)' 확인
!python scripts/train_lora.py --config configs/train_colab.yaml

In [ ]:
# 5. merge + 골든 평가 (train.yaml output_dir에서 라운드 자동 인식)
import yaml, os
cfg = yaml.safe_load(open('configs/train_colab.yaml'))
LORA_DIR = cfg['output_dir']
NAME = os.path.basename(LORA_DIR)
ROUND = NAME.split('-')[0]
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
ZIP = f'/content/lora_{ROUND}.zip'
print(f'round={ROUND}  lora={LORA_DIR}  eval={EVAL_JSON}  zip={ZIP}')
!python scripts/merge_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval data/eval/golden.jsonl --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

In [ ]:
# 6. zip + 다운로드 (브라우저로 바로 받음 — 세션 정리 전에 꼭 받기)
!zip -r {ZIP} {LORA_DIR} configs/train_colab.yaml configs/lora.yaml configs/model_qwen.yaml {EVAL_JSON}
!ls -la {ZIP}
from google.colab import files
files.download(ZIP)